In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"     # agama
os.environ["MKL_NUM_THREADS"] = "1"     # numpy, scipy
os.environ["OPENBLAS_NUM_THREADS"] = "1"    # numpy
os.environ["NUMEXPR_NUM_THREADS"] = "1"     # pandas

import agama
import torch 
import numpy as np
from scipy import integrate
# from scipy.stats import wasserstein_distance_nd
from astropy import units as u

from sbi.utils import BoxUniform

from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi


from sbi.utils import likelihood_nn

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import pickle
import matplotlib.pyplot as plt
from galaxy_generation import generate_galaxy_multiple, transform_params
from prior_generation import generate_prior
from standardization import standardize
from object_handler import load_csv

from model import prep_data

# from wasserstein_distance_nd import wasserstein_distance_nd 
import corner

torch.set_num_threads(4)


In [18]:
train_theta="./model_1/train_theta.csv"
train_x = "./model_1/train_x.csv"
t, x = prep_data(train_theta, train_x, standardization=True)

In [21]:
x.shape

AttributeError: 'tuple' object has no attribute 'shape'

In [3]:
def density(x: np.ndarray, 
            theta: torch.Tensor) -> np.ndarray:
    """
    Calculate density using GNFW profile as a function of r
    
    Params:
    - x: the value at which the function must be computed at (log_r_div_rstar)
    - theta: MCMC samples of the posterior
    """
    theta = np.array(transform_params(theta))
    
    alpha = theta[:,0]
    beta = theta[:,1]
    gamma = theta[:,2]
    p_0 = theta[:,3]
    r_s = theta[:,4]
    r_star = theta[:,5]
    
    r = 10 ** x * r_star

    rho = p_0 * (r / r_s) ** -gamma * (1 + (r / r_s) ** alpha) ** (-(beta-gamma)/alpha)
    
    return np.log10(rho)

In [8]:
cored_params = load_csv("mass_density_samples_model_14_core.csv", "Tensor")


In [11]:
(cored_params[:,1] + cored_params[:,0]).T


/tmp/ipykernel_1519478/3037505097.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /croot/pytorch-select_1717607455294/work/aten/src/ATen/native/TensorShape.cpp:3675.)
  (cored_params[:,1] + cored_params[:,0]).T


tensor([6.8871, 6.8602, 6.9556,  ..., 7.3751, 7.3311, 7.5828])

In [12]:
np.quantile(cored_params, [0.1,0.2], axis=0)

array([[ 7.19623461, -0.40459775, -0.86520606,  0.22706032],
       [ 7.34835968, -0.34937022, -0.74839115,  0.25332154]])